In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [39]:
root_path = "/home/stefan/ioai-prep/kits/pre-iaio/hidden-noise"

In [40]:
with open(f"{root_path}/custom_archive/starter_kit/model_params.json", "r") as f:
    params = json.load(f)

pipe = Pipeline([("scaler", StandardScaler()), ("regressor", LinearRegression())])

pipe.named_steps["scaler"].mean_ = np.array(params["scaler_mean"])
pipe.named_steps["scaler"].var_ = np.array(params["scaler_var"])
pipe.named_steps["scaler"].scale_ = np.array(params["scaler_scale"])
pipe.named_steps["scaler"].n_features_in_ = len(params["scaler_mean"])

pipe.named_steps["regressor"].coef_ = np.array(params["coef"])
pipe.named_steps["regressor"].intercept_ = params["intercept"]

pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None


# Data

In [41]:
train_df = pd.read_csv(f"{root_path}/train_data.csv")
train_df.head()

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,target,datapointID,subtaskID
0,38.386698,96.119755,74.799984,60.720818,17.224338,16.089440,7.296285,88.208061,60.737244,71.793142,-3.316884,0,1
1,7.509907,103.118333,92.599548,26.231124,27.665694,21.204378,39.120991,61.771641,46.851906,34.885306,-14.268649,1,1
2,57.722605,10.057386,23.272128,33.462029,39.583413,76.698472,14.443321,45.518765,56.918328,0.984856,-20.099994,2,1
3,62.298347,18.787687,9.154591,96.303772,99.248860,81.650804,32.924315,12.399853,69.459085,45.647168,-11.175629,3,1
4,14.072725,51.618305,6.646087,92.645215,29.129085,67.234057,34.152586,55.193712,55.924881,20.460944,-21.376248,4,1


In [42]:
X = train_df.drop(["target", "datapointID", "subtaskID"], axis=1)
pipe.score(X.values, train_df["target"])

0.822875539893386

# Deconstruction

In [43]:
scaler = pipe.named_steps["scaler"]
regressor = pipe.named_steps["regressor"]

mu = scaler.mean_
sigma = scaler.scale_
W = regressor.coef_
b = regressor.intercept_

In [44]:
W_norm = W / sigma
b_norm = b - np.sum((W * mu) / sigma)

In [ ]:
y = train_df["target"].values
X_noisy = X.values
W_norm_squared = np.sum(W_norm**2)

reconstructed = []
for i in range(len(X_noisy)):
    error = np.dot(W_norm, X_noisy[i]) + b_norm - y[i]
    correction = (error / W_norm_squared) * W_norm
    reconstructed.append(X_noisy[i] - correction)

X_reconstructed = np.array(reconstructed)

In [46]:
pipe.score(X_reconstructed, train_df["target"])

1.0

# Submission

In [54]:
def to_format(x):
    return str(round(x, 2))

submission_data = []
for idx, row_id in enumerate(train_df["datapointID"]):
    ans = ",".join(map(to_format, X_reconstructed[idx]))
    submission_data.append([1, row_id, ans])

In [55]:
submission_df = pd.DataFrame(
    submission_data, columns=["subtaskID", "datapointID", "answer"]
)

submission_df.head()

,subtaskID,datapointID,answer
0,1,0,"37.45,95.11,73.17,59.84,15.62,15.6,5.84,86.6,6..."
1,1,1,"2.02,97.19,83.1,21.07,18.26,18.37,30.61,52.36,..."
2,1,2,"61.21,13.82,29.31,36.74,45.56,78.5,19.85,51.5,..."
3,1,3,"60.74,17.11,6.46,94.84,96.59,80.85,30.51,9.73,..."
4,1,4,"12.19,49.59,3.39,90.87,25.9,66.26,31.23,51.97,..."


In [ ]:
submission_df.to_csv(f"{root_path}/submission.csv", index=False)